<a href="https://colab.research.google.com/github/mariolopezguasp/SP500Prediction/blob/main/1ModeloLineal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Modelo Lineal

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('dataset_ia_log_returns_10y.csv', index_col=0, parse_dates=True)

X = df.iloc[:-1].values
y = df.iloc[1:].values

n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Transform the validation and test data using the scaler fitted on training data
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train size: {X_train_scaled.shape}, Val size: {X_val_scaled.shape}, Test size: {X_test_scaled.shape}")

Train size: (1757, 12), Val size: (376, 12), Test size: (377, 12)


In [2]:

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import json

# Train a Linear Model (Ridge Regression to handle multicollinearity)
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

# Evaluate
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

mse_train = mean_squared_error(y_train, y_train_pred)
mse_val = mean_squared_error(y_val, y_val_pred)
mse_test = mean_squared_error(y_test, y_test_pred)

print(f"Train MSE: {mse_train:.6f}")
print(f"Validation MSE: {mse_val:.6f}")
print(f"Test MSE: {mse_test:.6f}")

results_linear = {'train_mse': mse_train, 'val_mse': mse_val, 'test_mse': mse_test, 'params': 'N/A'}
with open('results_linear.json', 'w') as f: json.dump(results_linear, f)


Train MSE: 0.000352
Validation MSE: 0.000268
Test MSE: 0.000318


In [3]:
rmse_train = np.sqrt(mse_train)
rmse_val = np.sqrt(mse_val)
rmse_test = np.sqrt(mse_test)

print("--- Root Mean Squared Error (RMSE) ---")
print(f"Train RMSE: {rmse_train:.6f} (Desviación media del modelo: {rmse_train*100:.2f}%)")
print(f"Validation RMSE: {rmse_val:.6f} (Desviación media del modelo: {rmse_val*100:.2f}%)")
print(f"Test RMSE: {rmse_test:.6f} (Desviación media del modelo: {rmse_test*100:.2f}%)")
print("-" * 38)

--- Root Mean Squared Error (RMSE) ---
Train RMSE: 0.018751 (Desviación media del modelo: 1.88%)
Validation RMSE: 0.016362 (Desviación media del modelo: 1.64%)
Test RMSE: 0.017820 (Desviación media del modelo: 1.78%)
--------------------------------------


I've applied `MinMaxScaler` to your `X_train`, `X_val`, and `X_test` datasets. The scaler was fitted only on the training data (`X_train`) to prevent data leakage from the validation and test sets. The scaled data is stored in `X_train_scaled`, `X_val_scaled`, and `X_test_scaled` respectively.

Now, when you train your model, you should use these scaled versions of your features.

In [6]:
import numpy as np

# --- Directional Accuracy for Training Set ---
signo_real_train = np.sign(y_train)
signo_pred_train = np.sign(y_train_pred)
accuracy_direccional_train = np.mean(signo_real_train == signo_pred_train)

# --- Directional Accuracy for Validation Set ---
signo_real_val = np.sign(y_val)
signo_pred_val = np.sign(y_val_pred)
accuracy_direccional_val = np.mean(signo_real_val == signo_pred_val)

# --- Directional Accuracy for Test Set ---
signo_real_test = np.sign(y_test)
signo_pred_test = np.sign(y_test_pred)
accuracy_direccional_test = np.mean(signo_real_test == signo_pred_test)

print("--- Accuracy Direccional ---")
print(f"Train Directional Accuracy: {accuracy_direccional_train*100:.2f}%")
print(f"Validation Directional Accuracy: {accuracy_direccional_val*100:.2f}%")
print(f"Test Directional Accuracy: {accuracy_direccional_test*100:.2f}%")

--- Accuracy Direccional ---
Train Directional Accuracy: 53.19%
Validation Directional Accuracy: 49.96%
Test Directional Accuracy: 51.95%


In [7]:
num_coeficientes = model.coef_.size
num_intercepto = 1 if model.fit_intercept else 0
num_parametros_totales = num_coeficientes + num_intercepto

print(f"El número de coeficientes (features) es: {num_coeficientes}")
print(f"¿El modelo tiene intercepto?: {bool(num_intercepto)}")
print(f"El número total de parámetros del modelo es: {num_parametros_totales}")

El número de coeficientes (features) es: 144
¿El modelo tiene intercepto?: True
El número total de parámetros del modelo es: 145
